# Single-tool eval: `query_prometheus_range`

Agent loop built from scratch (no `agent.agent` import), bound to `query_prometheus_range` **plus** its metric/label discovery helpers (`list_metric_names`, `get_label_values`) from `agent.tools.prometheus`. Everything outside this backend is still unavailable. This measures how far the model gets on this backend alone.

In [ ]:
from eval_transcript import REFUSAL, run_eval_repeats
from agent.tools.prometheus import get_label_values, list_metric_names, query_prometheus_range

# Runs per question. One is enough at temperature 0: the loop is
# deterministic, so repeats reproduce the same answer and the same cost.
RUNS = 1


In [ ]:
TOOLS = [list_metric_names, get_label_values, query_prometheus_range]

SYSTEM_PROMPT = """You are Oracle, a terse infrastructure-operations assistant for the Marconi lab.
You have 3 tools available, all read-only:
- `list_metric_names` — List available Prometheus metric names. Call this FIRST to discover what metrics exist before writing a query. `match` optionally filters by substring.
- `get_label_values` — List values for a Prometheus label such as 'service', 'job', or 'instance'. Use this to discover valid label values before querying.
- `query_prometheus_range` — Run a PromQL range query over the last `minutes`. Use for trends (e.g. CPU over the last 30m). The window is clamped to the configured maximum.
You are read-only. Never suggest you can restart or change anything yourself;
recommend the action for a human operator instead.
Call `list_metric_names` and `get_label_values` FIRST to confirm real values — never guess one in a `query_prometheus_range` query before confirming it exists.
If these tools can't answer the question, say so plainly instead of guessing.
Be concise."""

### The loop

Every run below goes through `eval_transcript.run_eval_loop`, shared across all the eval notebooks: it sends the messages, dispatches tool calls, prints each call's token counts, and — on every run — dumps the full transcript (every message sent, every response, every tool call/result, per-call token usage) to `eval/logs/<timestamp>__<label>.json`.

## In-scope question

A question the bound tools (`list_metric_names`, `get_label_values`, `query_prometheus_range`) together should be able to answer.

In [ ]:
summary = await run_eval_repeats(
    "05_query_prometheus_range:live-inscope",
    TOOLS,
    SYSTEM_PROMPT,
    "How has the 'up' metric trended over the last 30 minutes?",
    expect_tools=["query_prometheus_range", ("list_metric_names", "get_label_values")],
    runs=RUNS,
)


## Out-of-scope question

A question that needs evidence outside this backend. Watch whether the model admits the limit or hallucinates past it.

In [ ]:
summary = await run_eval_repeats(
    "05_query_prometheus_range:live-outofscope",
    TOOLS,
    SYSTEM_PROMPT,
    "What Prometheus metrics are available?",
    expect_tools=["list_metric_names"],
    runs=RUNS,
)


## Eval-dataset backend

The two runs above hit a live Prometheus over PromQL. `query_eval_metrics` (from `agent.tools.helper`) is a drop-in replacement backend: it streams the local eval fixture `data/eval/combined_metrics.jsonl` (all 3 machines' real metric samples, 2026-08-21) instead of live Prometheus — filtering by metric-name substring and/or machine rather than full PromQL. It's bound to the model the same way as the other tools — the model calls it, reads the raw JSON samples itself, and answers in plain English.

In [ ]:
from agent.tools.helper import query_eval_metrics

EVAL_TOOLS = [query_eval_metrics]
EVAL_SYSTEM_PROMPT = """You are Oracle, a terse infrastructure-operations assistant for the Marconi lab.
You have 1 tool available, read-only:
- `query_eval_metrics` — Search a local metrics dataset collected from 3 machines (marconi-01, marconi-02, lambda2) on 2026-08-21. Filter by `metric` (a substring to match in the metric name) and/or `machine`. Returns raw JSON metric samples, one per line.
Read the JSON samples the tool returns and answer the question in plain English —
never paste raw JSON back at the user.
You are read-only. Never suggest you can restart or change anything yourself;
recommend the action for a human operator instead.
If the tool can't answer the question, say so plainly instead of guessing.
Be concise."""

summary = await run_eval_repeats(
    "05_query_prometheus_range:eval",
    EVAL_TOOLS,
    EVAL_SYSTEM_PROMPT,
    "How did CPU usage behave on lambda2 over the day?",
    expect_tools=["query_eval_metrics"],
    runs=RUNS,
)
